In [ ]:
import pandas as pd
import re
from pathlib import Path


In [ ]:

# Directory with your CSVs
csv_dir = Path("/path/to/your/csvs")

# Patterns to remove (you can refine these later)
patterns_to_remove = [
    r"(?i)subscribe.*newsletter",
    r"(?i)follow.*(twitter|linkedin|facebook)",
    r"(?i)sign up.*(free|today)",
    r"(?i)welcome.*(back)?",
    r"(?i)hey (everyone|there|guys)",
    r"(?i)this post.*about",
    r"(?:https?://)?(?:www\.)?\S+\.\S+",  # remove links
]

def clean_text(text, patterns):
    for pattern in patterns:
        text = re.sub(pattern, "", text)
    return text.strip()

# Loop through all CSVs and clean
for csv_file in csv_dir.glob("*.csv"):
    df = pd.read_csv(csv_file)
    df['Cleaned_Content'] = df['Content'].apply(lambda x: clean_text(str(x), patterns_to_remove))
    df.to_csv(csv_dir / f"cleaned_{csv_file.name}", index=False)


In [ ]:
def deduplicate_sentences(text):
    sentences = list(dict.fromkeys(re.split(r'(?<=[.!?])\s+', text)))
    return ' '.join(sentences)


In [ ]:
!pip install pandas langid


In [ ]:
import numpy as np
import langid

EN_CODES = {"en"}  # extend if you also accept e.g. 'en-us', 'en-gb'

def _classify(text: str):
    """
    Returns (lang, confidence). Safely handles NaNs/short strings.
    """
    if not isinstance(text, str) or not text.strip():
        return "unk", 0.0
    lang, conf = langid.classify(text)
    return lang, conf

def find_non_english_rows(df: pd.DataFrame,
                          text_col: str,
                          min_conf: float = 0.90,
                          accept_langs: set = EN_CODES) -> pd.DataFrame:
    """
    Returns a *view* of rows that are NOT English (or below confidence threshold).

    Parameters
    ----------
    df : DataFrame
    text_col : column containing text to classify
    min_conf : minimum confidence to accept a prediction
    accept_langs : set of language codes considered 'English'
    """
    langs, confs = [], []
    for x in df[text_col].fillna(""):
        lang, conf = _classify(x)
        langs.append(lang)
        confs.append(conf)

    out = df.copy()
    out["__lang"] = langs
    out["__lang_conf"] = confs

    mask_non_en = ~out["__lang"].isin(accept_langs) | (out["__lang_conf"] < min_conf)
    return out.loc[mask_non_en, [*df.columns, "__lang", "__lang_conf"]]


def drop_non_english_rows(df: pd.DataFrame,
                          text_col: str,
                          min_conf: float = 0.90,
                          accept_langs: set = EN_CODES,
                          return_removed: bool = True):
    """
    Drops non‑English (or low‑confidence) rows and optionally also returns what was removed.

    Returns
    -------
    cleaned_df : DataFrame
    removed_df (optional) : DataFrame of dropped rows (with lang + conf columns)
    """
    # First detect them once
    non_en = find_non_english_rows(df, text_col, min_conf, accept_langs)

    # Build mask to keep English/high confidence rows
    keep_idx = df.index.difference(non_en.index)
    cleaned = df.loc[keep_idx].copy()

    if return_removed:
        return cleaned, non_en
    return cleaned


In [ ]:
df = pd.read_csv("your_file.csv")  # must contain e.g. a 'content' column

# 1) Just *see* the non‑English rows
non_en = find_non_english_rows(df, text_col="content", min_conf=0.90)
print(non_en.head())

# 2) Drop them
cleaned_df, removed_df = drop_non_english_rows(df, text_col="content", min_conf=0.90)
print(f"Kept: {len(cleaned_df)} | Removed: {len(removed_df)}")

cleaned_df.to_csv("clean_english_only.csv", index=False)
removed_df.to_csv("removed_non_english.csv", index=False)
